In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

# Question, input prompt
question = 'I just discovered the course. Can I join now?'

In [ ]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
# This is a simple API call to the LLM, later on we will add more to the logic of the function.
def llm(prompt):
    responses = openai_client.responses.create(
        model = 'llama-3.1-8b-instant',
        input = prompt
    )
    return responses.output_text

In [7]:
# Get the FAQ dataset
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [ ]:
from minsearch import Index

index = Index(
    # This is to indicate fields that may be helpful in determine the search
    text_fields=["question", "section", "answer"],
    # This is to indicate fields that require exact match (think of WHERE course = blabla) 
    # constrain the search within a subfield of the documents.
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
# The INSTRUCTIONS is the fixed part, it is the SYSTEM PROMPT
# that guides the LLM's behavior. It grounds the answer in our data and reduce hallucinations.
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

# The USER PROMPT template has the placeholder for question and retrieved context.
# The template is to be used with method .format() to fill in the question and context.
USER_PROMPT_TEMPLATE = """
Question: 
{question}

Context:
{context}
"""
# Build context, formated
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
# Retrieval relevant context from knowledge base:
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
        # To let the search prioritizr relevance found in one field over that from another
        boost_dict={'question': 2.0, 'section': 0.5},
        # This is the keyword_fields
        filter_dict={'course': course},
        num_results= 5
    )

# Build prompt
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

# RAG
def rag(question, course="llm-zoomcamp"):
    # Retrieve relevant contexts
    search_results = search(question, course="llm-zoomcamp")
    # Augment the context to the question
    user_prompt = build_prompt(question, search_result)
    # Input the augmented query to LLM
    return llm(user_prompt)

def llm(instructions, user_prompt, model='llama-3.1-8b-instant'):
    # This separation boosts the performance of the LLM,
    # Especially of those that during their fine-tuning process,
    # were trained to treat the System/Developer message as 
    # a "set of immutable rules or behavioral contraints"
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    # This clean, structured delimeter format also help with clarity for the model in parsing
    # Help the model to avoid "fuzzy" parsing, reduce its cognitive load.

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

def rag(query, model='llama-3.1-8b-instant'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

# From here, we use functions we built by importing them from packages housing them.

In [1]:
import os
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
